# `extract_distritos.ipynb` - Extracción de los polígonos de distritos (IDE Sevilla)

Requisito: *"Descarga directa del archivo GeoJSON oficial con la delimitación exacta de los 11 distritos de Sevilla desde el portal municipal (ideSEVILLA::distritos.geojson), garantizando geometrías cerradas y continuas."*

El fichero ya viene descargado en `data/distritos_sevilla.geojson` (descarga directa del portal IDE Sevilla). Este módulo lo carga con GeoPandas, valida la geometría y normaliza el nombre de cada distrito para poder cruzarlo de forma fiable con la tabla socioeconómica y con los puntos de recarga.

> Depende de `config.ipynb` (usa `RUTA_GEOJSON_DISTRITOS`, `CRS_GEOGRAFICO`).

In [ ]:
import logging
import re
import unicodedata

import geopandas as gpd

## Normalización de nombres de distrito

El GeoJSON usa guiones con espacios (`"San Pablo - Santa Justa"`), mientras que otras fuentes oficiales (Ayuntamiento, INE) usan guiones pegados (`"San Pablo-Santa Justa"`). `normalizar_nombre_distrito()` estandariza ambos formatos a una única clave, para poder cruzar las tres fuentes sin errores de coincidencia - el mismo patrón que `normalizar_nombre_pais()` en el proyecto GeoStat.

In [ ]:
def quitar_acentos(texto: str) -> str:
    """Elimina tildes/diacriticos ('Nervión' -> 'Nervion')."""
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def normalizar_nombre_distrito(nombre: str) -> str:
    """
    Estandariza el nombre de un distrito de Sevilla para poder cruzarlo
    entre el GeoJSON (IDE Sevilla), la tabla socioeconomica (Ayuntamiento/
    INE) y los puntos de recarga georreferenciados: colapsa los guiones
    con espacios a guiones pegados, elimina tildes y pasa a mayusculas.
    """
    if nombre is None:
        return ""
    limpio = re.sub(r"\s*-\s*", "-", nombre.strip())
    limpio = re.sub(r"\s+", " ", limpio)
    limpio = quitar_acentos(limpio).upper()
    return limpio

In [ ]:
# Prueba rápida
for ejemplo in ["San Pablo - Santa Justa", "Nervión", "Bellavista - La Palmera", "  Triana  "]:
    print(f"{ejemplo!r:32s} -> {normalizar_nombre_distrito(ejemplo)!r}")

## Función principal de extracción

In [ ]:
def extraer_distritos():
    """
    Carga el GeoJSON de distritos de Sevilla, valida que haya exactamente
    11 features con geometria valida, y añade la columna nombre_distrito
    (clave normalizada) para los cruces posteriores.
    """
    logger.info(f"Cargando GeoJSON de distritos desde: {RUTA_GEOJSON_DISTRITOS}")
    gdf = gpd.read_file(RUTA_GEOJSON_DISTRITOS)

    if gdf.crs is None:
        logger.warning("El GeoJSON no declara CRS; se asume EPSG:4326 (WGS84).")
        gdf = gdf.set_crs(CRS_GEOGRAFICO)
    elif str(gdf.crs) != CRS_GEOGRAFICO:
        logger.info(f"Reproyectando distritos de {gdf.crs} a {CRS_GEOGRAFICO}")
        gdf = gdf.to_crs(CRS_GEOGRAFICO)

    n_distritos = len(gdf)
    if n_distritos != 11:
        logger.warning(f"Se esperaban 11 distritos, se han leido {n_distritos}.")

    # Geometrias cerradas y continuas: se reparan geometrias invalidas
    # (autointersecciones, anillos abiertos) con el truco estandar buffer(0)
    n_invalidas = (~gdf.geometry.is_valid).sum()
    if n_invalidas > 0:
        logger.warning(f"Geometrias invalidas detectadas: {n_invalidas}. Reparando con buffer(0).")
        gdf["geometry"] = gdf.geometry.buffer(0)
    else:
        logger.info("Las 11 geometrias son validas (cerradas y continuas).")

    gdf["nombre_distrito"] = gdf["Distri_11D"].apply(normalizar_nombre_distrito)

    logger.info(f"Distritos cargados: {n_distritos} | CRS: {gdf.crs}")
    return gdf

## Prueba rápida

Carga real del fichero, sin depender de ningún otro módulo aparte de `config.ipynb` y del `logger` (si se abre este notebook de forma aislada, ejecuta antes esas dos celdas `%run`).

In [ ]:
gdf_distritos_prueba = extraer_distritos()
print(gdf_distritos_prueba[["nombre_distrito", "Area"]].sort_values("nombre_distrito").to_string(index=False))

---
✅ **Extracción de distritos verificada**: 11 polígonos, geometrías válidas, nombres normalizados.